# 過去JRAレース自動収集

このノートブックは、GitHubの最新版を取得し、過去レース結果をGoogle Driveへ保存します。実行セルは1つです。


In [ ]:
import importlib
from pathlib import Path
import shutil
import subprocess
import sys

from google.colab import drive
from IPython.display import display

GITHUB_REPO = globals().get('GITHUB_REPO', 'f94wzpjvr2-stack/keiba-analysis')
GITHUB_BRANCH = globals().get('GITHUB_BRANCH', 'main')
PROJECT_DIR = globals().get('PROJECT_DIR', '/content/keiba-analysis')
DRIVE_DATA_DIR = globals().get('DRIVE_DATA_DIR', '/content/drive/MyDrive/keiba-ev-data')
RACE_COUNT = 100
# 通常は空欄で開始。自動探索に失敗した場合だけ、最近のJRAレース結果URLを1件入れます。
JRA_SEED_URLS = []

drive.mount('/content/drive')
project = Path(PROJECT_DIR)
if project.exists():
    shutil.rmtree(project)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH,
    f'https://github.com/{GITHUB_REPO}.git', PROJECT_DIR,
], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{PROJECT_DIR}[dev]'
], check=True)
if f'{PROJECT_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{PROJECT_DIR}/src')

import_recent_jra_races = importlib.import_module(
    'keiba_ev.historical'
).import_recent_jra_races

summary = import_recent_jra_races(
    race_count=RACE_COUNT,
    output_dir=DRIVE_DATA_DIR,
    seed_urls=JRA_SEED_URLS or None,
    resume=True,
    request_interval=0.8,
)
display(summary)
print('保存ファイル:')
for path in sorted(Path(DRIVE_DATA_DIR).glob('historical_*.csv')):
    print(' -', path.name)
